# Cloud-Adapter HRC_WHU development pilot

Run in **Google Colab with a GPU runtime**. This evaluates the official binary Cloud-Adapter checkpoint trained on HRC_WHU on exactly three dev patches: one cloud-free Brazil patch and two Hurricane Helene patches with different reviewed cloud coverage. It does not read test labels or tune model parameters.

Each 1024 x 1024 RGB patch is split into sixteen non-overlapping native-resolution 256 x 256 crops to match the HRC_WHU training crop size. The runner passes uint8 BGR values through MMSegmentation's configured preprocessor, which converts to RGB and applies the configured mean/std. This differs from the Cloud-Adapter demo's per-image min-max scaling. Outputs are preliminary: three patches, initial single-reviewer labels, and HRC_WHU's 0.5-15 m Google Earth imagery is closer in resolution to Maxar but not an exact sensor/domain match. The source dataset is for academic use; review the dataset terms before other uses.

In [ ]:
import subprocess
from pathlib import Path

if not Path('/content').exists():
    raise RuntimeError('This pilot is designed for Google Colab.')
print(subprocess.run(['nvidia-smi'], check=True, capture_output=True, text=True).stdout[:2000])

## Install inference dependencies

The Cloud-Adapter implementation depends on compiled MMCV operators and a compatible PyTorch/CUDA build. Its checkpoint is about 1.32 GB; allow time for installation and download.

In [ ]:
%pip -q install openmim huggingface_hub
!mim install -q mmengine==0.10.7
!mim install -q mmcv==2.1.0
%pip -q install mmsegmentation==1.2.2 mmdet==3.3.0

In [ ]:
import subprocess
from pathlib import Path

PROJECT_REPO = 'https://github.com/Break-Through-Tech/Ursa-Space-Systems-1D-cloud-detection-in-very-high-resolution-optical-satellite-imagery.git'
CLOUD_ADAPTER_REPO = 'https://github.com/XavierJiezou/Cloud-Adapter.git'
CLOUD_ADAPTER_COMMIT = '78c7ea50611b2381f45206a4269a2346116f4525'
PROJECT_DIR = Path('/content/ursa-cloud-benchmark')
CLOUD_ADAPTER_DIR = Path('/content/Cloud-Adapter')

if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO, str(PROJECT_DIR)], check=True)
if not CLOUD_ADAPTER_DIR.exists():
    subprocess.run(['git', 'clone', CLOUD_ADAPTER_REPO, str(CLOUD_ADAPTER_DIR)], check=True)
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CLOUD_ADAPTER_COMMIT], cwd=CLOUD_ADAPTER_DIR, check=True)
subprocess.run(['git', 'checkout', '--detach', CLOUD_ADAPTER_COMMIT], cwd=CLOUD_ADAPTER_DIR, check=True)

EVAL_ROOT = PROJECT_DIR / 'outputs' / 'reviewed_eval_set'
if not (EVAL_ROOT / 'split_manifest.csv').is_file():
    subprocess.run(['python', 'notebooks/prepare_reviewed_eval.py'], cwd=PROJECT_DIR, check=True)
print('Evaluation patches:', EVAL_ROOT)

In [ ]:
import sys
import torch
from huggingface_hub import hf_hub_download

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Colab GPU runtime before running inference.')

sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))
from pipeline.evaluate_cloud_adapter_pilot import run_pilot

MODEL_REPO = 'XavierJiezou/cloud-adapter-models'
MODEL_REVISION = 'cf76783e0696f014e251f71d1b42f74b9bf0f0f6'
CHECKPOINT_NAME = 'checkpoints/cloud-adapter/hrc_whu_full_weight.pth'
CHECKPOINT = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=CHECKPOINT_NAME,
    revision=MODEL_REVISION,
)
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'cloud_adapter_hrc_whu_pilot'
print('Checkpoint size (GB):', round(Path(CHECKPOINT).stat().st_size / 1e9, 2))

## Run and inspect the pilot

The script reports per-patch and pooled binary precision, recall, F1, and IoU, saves masks and comparison panels, and records the model revision and preprocessing metadata under `outputs/cloud_adapter_hrc_whu_pilot/`.

In [ ]:
records, pooled_metrics, run_metadata = run_pilot(
    eval_root=EVAL_ROOT,
    output_root=OUTPUT_DIR,
    cloud_adapter_root=CLOUD_ADAPTER_DIR,
    checkpoint_path=CHECKPOINT,
)
print('Per-patch metrics:')
for record in records:
    print(record)
print('Pooled preliminary metrics:', pooled_metrics)
print('Saved predictions and metadata to:', OUTPUT_DIR)